# Mushroom Classification V3
### Key fixes over V2:
1. odor missing kept as a separate category (not mode-imputed) — avoids corrupting the #1 feature
2. stalk-root 'missing' kept as category — it carries real signal
3. New interaction features — odor x gill-color, odor x spore-print-color, etc.
4. odor_is_missing flag — lets model learn from the missingness itself
5. Retrain final model on full data using best params

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier, AdaBoostClassifier, VotingClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import os
print('Libraries loaded')

## 1. Data Loading

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train = pd.read_csv('/kaggle/input/mlp-jan-2026-assignment-2/train.csv')
test  = pd.read_csv('/kaggle/input/mlp-jan-2026-assignment-2/test.csv')
sample_submission = pd.read_csv('/kaggle/input/mlp-jan-2026-assignment-2/sample_submission.csv')

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')
train.head()

## 2. Missing Value Handling

Categorical NaN values are filled with 'unknown' so the model treats missingness as its own category rather than imputing a fake value. Numerical NaN values use median from train. The string 'missing' in stalk-root is kept as-is since it carries real signal.

In [ ]:
def handle_missing_v3(df, ref_df=None, skip_cols=['ID', 'mushroom_id', 'class']):
    """
    V3 strategy:
    - Categorical NaN → fill with 'unknown' (preserve as new category, not imputed)
    - Numerical NaN   → median from train (same as before)
    - Do NOT convert 'missing' string in stalk-root — keep it as a valid category
    """
    df  = df.copy()
    src = ref_df if ref_df is not None else df

    cat_cols = [c for c in df.columns
                if (df[c].dtype == object or str(df[c].dtype) == 'str')
                and c not in skip_cols]
    
    num_cols = [c for c in df.columns
                if df[c].dtype in [np.int64, np.float64]
                and c not in skip_cols]

    # Categorical NaN → 'unknown'
    for col in cat_cols:
        n = df[col].isna().sum()
        if n > 0:
            df[col] = df[col].fillna('unknown')
            print(f'  [Cat] {col}: filled {n} NaN → "unknown"')

    # Numerical NaN → median from train
    for col in num_cols:
        n = df[col].isna().sum()
        if n > 0:
            median_val = src[col].median()
            df[col] = df[col].fillna(median_val)
            print(f'  [Num] {col}: filled {n} NaN → median={median_val}')

    return df

print('Cleaning train')
train = handle_missing_v3(train)
print('Cleaning test')
test  = handle_missing_v3(test, ref_df=train)

print(f'Remaining NaN in train: {train.isnull().sum().sum()}')
print(f'Remaining NaN in test : {test.isnull().sum().sum()}')

## 3. Duplicate Detection

In [ ]:
feature_cols = [c for c in train.columns if c not in ['ID', 'mushroom_id']]
dup_feature  = train.duplicated(subset=feature_cols).sum()
print(f'Feature duplicates: {dup_feature}')
if dup_feature > 0:
    train = train.drop_duplicates(subset=feature_cols).reset_index(drop=True)
    print(f'New train shape: {train.shape}')
else:
    print('No duplicates found')

## 4. Feature Engineering

Interaction features between the strongest predictors. Tree models can discover these on their own but making them explicit reduces the depth needed and often improves test generalization.

In [ ]:
def add_features(df):
    df = df.copy()
    
    # Missingness flags
    df['odor_is_unknown']       = (df['odor'] == 'unknown').astype(int)
    df['stalkroot_is_missing']  = (df['stalk-root'] == 'missing').astype(int)
    df['stalkroot_is_unknown']  = (df['stalk-root'] == 'unknown').astype(int)
    
    # Interaction features
    df['odor_gillcolor']        = df['odor'].astype(str) + '_' + df['gill-color'].astype(str)
    df['odor_spore']            = df['odor'].astype(str) + '_' + df['spore-print-color'].astype(str)
    df['spore_gillcolor']       = df['spore-print-color'].astype(str) + '_' + df['gill-color'].astype(str)
    df['odor_habitat']          = df['odor'].astype(str) + '_' + df['habitat'].astype(str)
    df['capcolor_odor']         = df['cap-color'].astype(str) + '_' + df['odor'].astype(str)
    
    return df

train = add_features(train)
test  = add_features(test)

new_feat_cols = ['odor_is_unknown', 'stalkroot_is_missing', 'stalkroot_is_unknown',
                 'odor_gillcolor', 'odor_spore', 'spore_gillcolor', 'odor_habitat', 'capcolor_odor']
print(f'Added {len(new_feat_cols)} new features: {new_feat_cols}')
print(f'Train shape: {train.shape}')

## 5. Encoding & Scaling

In [ ]:
# Target encoding
y = train['class'].map({'e': 0, 'p': 1}).values

drop_cols = ['ID', 'mushroom_id', 'class']
X_raw     = train.drop(columns=drop_cols)
test_id   = test['ID']
Xt_raw    = test.drop(columns=['ID', 'mushroom_id'])

cat_feat = [c for c in X_raw.columns if X_raw[c].dtype == object or str(X_raw[c].dtype) == 'str']
num_feat = [c for c in X_raw.columns if X_raw[c].dtype in [np.int64, np.float64]]

print(f'Categorical features ({len(cat_feat)}): {cat_feat}')
print(f'Numerical features  ({len(num_feat)}): {num_feat}')

In [ ]:
# Combine train+test for unified label encoding (tree models)
combined = pd.concat([X_raw, Xt_raw], axis=0).reset_index(drop=True)
n_train  = len(X_raw)

le_dict = {}
for col in cat_feat:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))
    le_dict[col]  = le
    print(f'  {col}: {len(le.classes_)} categories encoded')

X            = combined.iloc[:n_train].copy().reset_index(drop=True)
X_test_final = combined.iloc[n_train:].copy().reset_index(drop=True)

# OneHotEncoding + StandardScaler for distance-based models (LR, KNN, SVM, NB)
# This matches friend's approach and gives much better signal for these models
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_feat),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_feat)
])
X_ohe      = preprocessor.fit_transform(X_raw)
X_test_ohe = preprocessor.transform(Xt_raw)

# Scaled label encoded (kept for compatibility)
scaler              = StandardScaler()
X_sc                = X.copy()
X_test_sc           = X_test_final.copy()
X_sc[num_feat]      = scaler.fit_transform(X[num_feat])
X_test_sc[num_feat] = scaler.transform(X_test_final[num_feat])

# Train-Val split — same indices for all representations
idx = np.arange(len(X))
idx_tr, idx_vl = train_test_split(idx, test_size=0.2, random_state=42, stratify=y)

X_tr,     X_vl     = X.iloc[idx_tr],      X.iloc[idx_vl]
X_tr_sc,  X_vl_sc  = X_sc.iloc[idx_tr],   X_sc.iloc[idx_vl]
X_tr_ohe, X_vl_ohe = X_ohe[idx_tr],       X_ohe[idx_vl]
y_tr,     y_vl     = y[idx_tr],            y[idx_vl]

print(f'Label encoded : {X_tr.shape}')
print(f'OHE encoded   : {X_tr_ohe.shape}')

## 6. Model Building

In [ ]:
results = {}

def evaluate(name, model, Xtr, ytr, Xv, yv):
    model.fit(Xtr, ytr)
    preds = model.predict(Xv)
    acc   = accuracy_score(yv, preds)
    cv    = cross_val_score(model, Xtr, ytr, cv=5, scoring='accuracy').mean()
    results[name] = {'Val Accuracy': round(acc, 5), '5-Fold CV': round(cv, 5)}
    print(f'{name:<38} Val={acc:.5f}  CV={cv:.5f}')
    return model

print(f'{"Model":<38} {"Val Accuracy":<18} 5-Fold CV')
print('-'*65)

lr   = evaluate('1. Logistic Regression',    LogisticRegression(max_iter=1000, random_state=42),                                         X_tr_ohe, y_tr, X_vl_ohe, y_vl)
dt   = evaluate('2. Decision Tree',          DecisionTreeClassifier(random_state=42),                                                    X_tr,     y_tr, X_vl,     y_vl)
nb   = evaluate('3. Naive Bayes',            GaussianNB(),                                                                               X_tr_ohe, y_tr, X_vl_ohe, y_vl)
knn  = evaluate('4. K-Nearest Neighbors',    KNeighborsClassifier(n_neighbors=5, n_jobs=-1),                                             X_tr_ohe, y_tr, X_vl_ohe, y_vl)
svm  = evaluate('5. SVM (RBF)',              SVC(kernel='rbf', random_state=42),                                                         X_tr_ohe, y_tr, X_vl_ohe, y_vl)
ada  = evaluate('6. AdaBoost',               AdaBoostClassifier(n_estimators=200, random_state=42),                                      X_tr,     y_tr, X_vl,     y_vl)
rf   = evaluate('7. Random Forest',          RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),                       X_tr,     y_tr, X_vl,     y_vl)
gb   = evaluate('8. Gradient Boosting',      GradientBoostingClassifier(n_estimators=200, random_state=42),                              X_tr,     y_tr, X_vl,     y_vl)
et   = evaluate('9. Extra Trees',            ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1),                         X_tr,     y_tr, X_vl,     y_vl)
xgb  = evaluate('10. XGBoost',               XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, eval_metric='logloss', random_state=42, n_jobs=-1), X_tr, y_tr, X_vl, y_vl)
lgbm = evaluate('11. LightGBM',              LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1),           X_tr, y_tr, X_vl, y_vl)

## 7. Hyperparameter Tuning

In [ ]:
# Tuning 1: Random Forest
print('Tuning Random Forest')
rf_params = {
    'n_estimators'     : [200, 300, 500],
    'max_depth'        : [None, 10, 20, 30],
    'min_samples_split': [2, 5],
    'max_features'     : ['sqrt', 'log2']
}
rf_gs = RandomizedSearchCV(RandomForestClassifier(random_state=42, n_jobs=1),
                            rf_params, n_iter=10, cv=3, scoring='accuracy',
                            random_state=42, n_jobs=-1, verbose=1)
rf_gs.fit(X_tr, y_tr)
rf_tuned_acc = accuracy_score(y_vl, rf_gs.best_estimator_.predict(X_vl))
print(f'Best params : {rf_gs.best_params_}')
print(f'CV Score    : {rf_gs.best_score_:.5f}')
print(f'Val Accuracy: {rf_tuned_acc:.5f}')
results['RF Tuned'] = {'Val Accuracy': round(rf_tuned_acc, 5), '5-Fold CV': round(rf_gs.best_score_, 5)}

In [ ]:
# Tuning 2: XGBoost
print('Tuning XGBoost')
xgb_params = {
    'n_estimators'     : [200, 300, 500],
    'max_depth'        : [4, 6, 8],
    'learning_rate'    : [0.01, 0.05, 0.1],
    'subsample'        : [0.8, 1.0],
    'colsample_bytree' : [0.8, 1.0]
}
xgb_gs = RandomizedSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=1),
    xgb_params, n_iter=10, cv=3, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1)
xgb_gs.fit(X_tr, y_tr)
xgb_tuned_acc = accuracy_score(y_vl, xgb_gs.best_estimator_.predict(X_vl))
print(f'Best params : {xgb_gs.best_params_}')
print(f'CV Score    : {xgb_gs.best_score_:.5f}')
print(f'Val Accuracy: {xgb_tuned_acc:.5f}')
results['XGBoost Tuned'] = {'Val Accuracy': round(xgb_tuned_acc, 5), '5-Fold CV': round(xgb_gs.best_score_, 5)}

In [ ]:
# Tuning 3: LightGBM
print('Tuning LightGBM')
lgbm_params = {
    'n_estimators' : [200, 300],
    'max_depth'    : [4, 6, -1],
    'learning_rate': [0.05, 0.1],
    'num_leaves'   : [31, 63],
    'subsample'    : [0.8, 1.0]
}
lgbm_gs = RandomizedSearchCV(
    LGBMClassifier(random_state=42, n_jobs=1, verbose=-1),
    lgbm_params, n_iter=10, cv=3, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1)
lgbm_gs.fit(X_tr, y_tr)
lgbm_tuned_acc = accuracy_score(y_vl, lgbm_gs.best_estimator_.predict(X_vl))
print(f'Best params : {lgbm_gs.best_params_}')
print(f'CV Score    : {lgbm_gs.best_score_:.5f}')
print(f'Val Accuracy: {lgbm_tuned_acc:.5f}')
results['LightGBM Tuned'] = {'Val Accuracy': round(lgbm_tuned_acc, 5), '5-Fold CV': round(lgbm_gs.best_score_, 5)}

## 8. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'Model'})
results_df = results_df.sort_values('Val Accuracy', ascending=False).reset_index(drop=True)
print('Model comparison')
print(results_df.to_string(index=False))

## 9. Soft Voting Ensemble

In [ ]:
print('Ensemble: Soft Voting (RF + XGB + LGBM)')
ensemble = VotingClassifier(
    estimators=[('rf',   rf_gs.best_estimator_),
                ('xgb',  xgb_gs.best_estimator_),
                ('lgbm', lgbm_gs.best_estimator_)],
    voting='soft', n_jobs=-1
)
ensemble.fit(X_tr, y_tr)
ens_acc = accuracy_score(y_vl, ensemble.predict(X_vl))
print(f'Ensemble Val Accuracy: {ens_acc:.5f}')
results['Ensemble (RF+XGB+LGBM)'] = {'Val Accuracy': round(ens_acc, 5), '5-Fold CV': '-'}

## 10. Final Submission

Retrain ensemble on full training data using best hyperparams.

In [ ]:
print('Retraining on full training data')

rf_f   = RandomForestClassifier(**rf_gs.best_params_,   random_state=42, n_jobs=-1)
xgb_f  = XGBClassifier(**xgb_gs.best_params_,  eval_metric='logloss', random_state=42, n_jobs=-1)
lgbm_f = LGBMClassifier(**lgbm_gs.best_params_, random_state=42, n_jobs=-1, verbose=-1)

# Friend's winning approach: GradientBoosting on full OHE data
from sklearn.ensemble import GradientBoostingClassifier
gb_full = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42)
gb_full.fit(X_ohe, y)
gb_val_acc = accuracy_score(y_vl, gb_full.predict(X_vl_ohe))
print(f'GradientBoosting (OHE, full train) val: {gb_val_acc:.5f}')

# Auto-select best model
ens_val          = results.get('Ensemble (RF+XGB+LGBM)', {}).get('Val Accuracy', 0)
best_single_name = max(
    ['RF Tuned', 'XGBoost Tuned', 'LightGBM Tuned'],
    key=lambda k: results.get(k, {}).get('Val Accuracy', 0)
)
best_single_acc = results.get(best_single_name, {}).get('Val Accuracy', 0)
best_acc = max(ens_val, best_single_acc, gb_val_acc)

if gb_val_acc == best_acc:
    print(f'Using GradientBoosting OHE (val={gb_val_acc:.5f})')
    final_model  = gb_full
    use_ohe      = True
elif ens_val == best_acc:
    print(f'Using ensemble (val={ens_val:.5f})')
    final_model = VotingClassifier(
        estimators=[('rf', rf_f), ('xgb', xgb_f), ('lgbm', lgbm_f)],
        voting='soft', n_jobs=-1
    )
    final_model.fit(X, y)
    use_ohe = False
else:
    print(f'Using {best_single_name} (val={best_single_acc:.5f})')
    model_map   = {'RF Tuned': rf_f, 'XGBoost Tuned': xgb_f, 'LightGBM Tuned': lgbm_f}
    final_model = model_map[best_single_name]
    final_model.fit(X, y)
    use_ohe = False

print('Done')

preds_num = final_model.predict(X_test_ohe if use_ohe else X_test_final)
preds_lbl = np.where(preds_num == 0, 'e', 'p')

submission = pd.DataFrame({'ID': test_id.values, 'class': preds_lbl})
submission.to_csv('submission.csv', index=False)
print(f'submission.csv saved! Shape: {submission.shape}')
print(f"Edible: {sum(preds_lbl=='e')} | Poisonous: {sum(preds_lbl=='p')}")
submission.head(10)


## 11. Feature Importance

In [ ]:
fi = pd.DataFrame({'Feature': X_tr.columns,
                   'Importance': lgbm_gs.best_estimator_.feature_importances_})
fi = fi.sort_values('Importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(fi['Feature'][::-1], fi['Importance'][::-1], color='steelblue', edgecolor='black')
ax.set_title('Top 20 Feature Importances (LightGBM Tuned)', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance_v3.png', dpi=120)
plt.show()